# Transformer Operations

This notbook follow the Transformer Ops lecture by Mohsena (AI by Hand) [Link](https://www.byhand.ai/p/library-videos-counting-ai-by-hand-transformer-ops).

It covers the foundational layer of operational cost of the transformer architecture. I recommend the availabe Excel worksheets, this implementation is on python.

In [98]:
# Dependencies
import numpy as np
import torch

# np.random.seed(seed=123)

### What are FLOPs?

Floating Poing Operation (FLOP)

Plural -> FLOPs

Measure per seconds -> FLOPS

### Why do FLOP Matters

Under the hood, Neural Networks (NN) are Matrix operations. SOTA models perform billions of these operations, with Matrix Multiplication dominating FLOPs.

In [99]:
# Random integer
a = np. random.randint(11) # Half-open interval [)
b = np.random.randint(11)

print(f"Random integer a: {a}")
print(f"Random integer b: {b}")

Random integer a: 8
Random integer b: 7


In [100]:
print("Summation corresponds to 1 FLOP")
print(f"Sum of a and b: {a + b}.")
print("Multiplication corresponds to 1 FLOP")
print(f"Product of a and b: {a * b}")


Summation corresponds to 1 FLOP
Sum of a and b: 15.
Multiplication corresponds to 1 FLOP
Product of a and b: 56


In [101]:
# Random vector
vec_a = np.random.randint(11, size=(1,3))
vec_b = np.random.randint(11, size=(1,3))

print(f"Random vector vec_a: {vec_a}, dimension d={vec_a.shape[1]}")
print(f"Random vector vec_b: {vec_b}, dimension d={vec_b.shape[1]}")

Random vector vec_a: [[0 1 5]], dimension d=3
Random vector vec_b: [[5 2 1]], dimension d=3


In [102]:
print("Vector summention corresponds to d FLOPs")
print(f"Sum of vec_a and vec_b: {vec_a + vec_b}. Therefore, {vec_a.size} FLOPs")
print("Vector multiplication corresponds to d+(d-1) FLOPs")
print(f"Multiplication of vec_a and vec_b: {vec_a*vec_b}. Therefore, {vec_a.size + (vec_a.size-1)} FLOPs")

Vector summention corresponds to d FLOPs
Sum of vec_a and vec_b: [[5 3 6]]. Therefore, 3 FLOPs
Vector multiplication corresponds to d+(d-1) FLOPs
Multiplication of vec_a and vec_b: [[0 2 5]]. Therefore, 5 FLOPs


In [103]:
# Random Matrix
mat_a = np.random.randint(11, size=(2,3))

print("Random Matrix mat_a:")
print(mat_a)
print(f"Rows: r={mat_a.shape[0]}, columns: c={mat_a.shape[1]}")

Random Matrix mat_a:
[[10  4  6]
 [ 9  7  1]]
Rows: r=2, columns: c=3


In [104]:
print("Matrix-vector multiplication correspond to r*(2c-1) FLOPs")
print(f"Multiplication of mat_a {mat_a.shape} and vec_a.T {vec_a.T.shape}:")
print(mat_a@vec_a.T)
print(f"With shape: {(mat_a@vec_a.T).shape}")
print(f"Therefore, {(mat_a.shape[0]*(2*mat_a.shape[1]-1))} FLOPs")

Matrix-vector multiplication correspond to r*(2c-1) FLOPs
Multiplication of mat_a (2, 3) and vec_a.T (3, 1):
[[34]
 [12]]
With shape: (2, 1)
Therefore, 10 FLOPs


In [105]:
mat_b = np.random.randint(11, size=(3,4))

print("Random Matrix mat_b:")
print(mat_b)
print(f"Rows: r={mat_b.shape[0]}, columns: c={mat_b.shape[1]}")

Random Matrix mat_b:
[[ 6  3 10 10]
 [ 6  0  7  6]
 [ 6  0  3  5]]
Rows: r=3, columns: c=4


In [106]:
print("Matrix-Matrix multiplication correspond to r1*(c2*(2*c1-1))) FLOPs")
print(f"Multiplication of mat_a {mat_a.shape} and mat_b {mat_b.shape}:")
print(mat_a@mat_b)
print(f"With shape: {(mat_a@mat_b).shape}")
print(f"Therefore, {(mat_a.shape[0]*(mat_b.shape[1]*(2*mat_a.shape[1]-1)))} FLOPs")

Matrix-Matrix multiplication correspond to r1*(c2*(2*c1-1))) FLOPs
Multiplication of mat_a (2, 3) and mat_b (3, 4):
[[120  30 146 154]
 [102  27 142 137]]
With shape: (2, 4)
Therefore, 40 FLOPs


It is clear that FLOPs rapidly increase with larger matrices

In [107]:
mat_c = np.random.randint(11, size=(12,10))
mat_d = np.random.randint(11, size=(10,20))

print(f"Random Matrix mat_c shape: {mat_c.shape}")
print(f"Random Matrix mat_d shape: {mat_d.shape}")

mat_e = mat_c@mat_d

print(f"Matrix mat_e = mat_c@mat_d, with shape: {mat_e.shape}")
print(f"FLOPs = {mat_c.shape[0]*(mat_d.shape[1]*(mat_c.shape[1]+(mat_c.shape[1]-1)))}")

Random Matrix mat_c shape: (12, 10)
Random Matrix mat_d shape: (10, 20)
Matrix mat_e = mat_c@mat_d, with shape: (12, 20)
FLOPs = 4560


## Flow of the Transformer Architecture

Overview

Input token embeddings -> Attention -> Feed Forward Network (FNN) -> Output

Input -> Q,K,V projections -> Attention Matrix -> Output projection -> FFN -> Output

Total FLOPs = Attention FLOPs + FFN FLOPs

### IMPORTANT PARAMETERS

1. Sequence length (n): tokens processed at once.
2. Embedding Size (d): Size of the vector representation of each token. Larger d provides a richer representation of the token, but increases the computation O(d^2).
3. Number of heads (h): Instead of one large attention system, we split attention into multiple heads 
4. Hidden dimension (hd): dimension of projection matrices
5. Number of layers (L): Repeated processing block

Each layer contains -> Attention + FNN

Transformer FLOPs = L x Layer FLOPs

In [109]:
def matmul_flops_count(mat1_shape, mat2_shape):
    r1, c1 = mat1_shape
    r2, c2 = mat2_shape

    if c1 != r2:
        raise ValueError("Matrix dimensions are not compatible for multiplication.")
    else:
        return r1 * (c2 * (2 * c1 - 1))

In [121]:
n = 6
d = 5
h = 6
hd = 3
L = 3



# Input
x = np.random.randint(4, size=(d,n))
print(x)
print(f"Input shape: {x.shape}")

[[0 0 1 3 2 0]
 [0 1 3 1 3 1]
 [1 3 3 2 1 0]
 [0 3 2 1 2 3]
 [3 3 2 0 0 0]]
Input shape: (5, 6)


### Attention Projection FLOPs

Attention requires 3 projection Matrices: W_q, W_k, and W_v

Q = W_q@x

K = W_k@x

V = W_v@x

In [122]:
W_q = np.random.randint(-1, 2, size=(hd,d))
W_k = np.random.randint(-1, 2, size=(hd,d))
W_v = np.random.randint(-1, 2, size=(hd,d))

print(f"Projection Matrices shape: {W_q.shape}")


Projection Matrices shape: (3, 5)


In [123]:
Q = W_q@x
K = W_k@x
V = W_v@x

proj_flops = matmul_flops_count(W_q.shape, x.shape)
att_proj_flops = 3 * proj_flops
mha_proj_flops = h * att_proj_flops

print(f"Attention Matrices (Q,K,V) shape: {Q.shape}")
print(f"Single projection FLOPs = {proj_flops}")
print(f"Total Projections FLOPs = 3 * Projection FLOPs: {att_proj_flops}")
print(f"MHA projections FLOPs = h * total projection FLOPs : {mha_proj_flops}")

Attention Matrices (Q,K,V) shape: (3, 6)
Single projection FLOPs = 162
Total Projections FLOPs = 3 * Projection FLOPs: 486
MHA projections FLOPs = h * total projection FLOPs : 2916


### Attention FLOPs

Attention output computation requires 3 steps:

1. Compute the attention matrix: A = Q@K^T/sqrt(d_k)
2. Apply softmax to the attention matrix: A' = softmax(A)
3. Compute the output: O = A'@V

